In [ ]:
# Install required libraries

!pip install -q langchain
!pip install -q langgraph
!pip install -q faiss-cpu
!pip install -q google-generativeai
!pip install -q langchain-google-genai
!pip install -q pypdf
!pip install -q faker
!pip install -q streamlit
!pip install -q pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 87.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/FinAssist-AI"

folders = [
    "data",
    "data/policies",
    "data/sample_pdfs",
    "outputs",
]

for folder in folders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [ ]:
import pandas as pd
import random
from faker import Faker

fake = Faker()

data = []

for i in range(100):

    credit_score = random.randint(550, 850)
    income = random.randint(30000, 150000)
    expenses = random.randint(1000, 7000)
    loan_amount = random.randint(5000, 500000)

    employment_type = random.choice([
        "Salaried",
        "Self-Employed"
    ])

    repayment_history = random.choice([
        "Good",
        "Average",
        "Poor"
    ])

    years_employed = random.randint(1, 15)

    data.append({
        "applicant_id": f"APP-{1000+i}",
        "name": fake.name(),
        "age": random.randint(21, 60),
        "employment_type": employment_type,
        "annual_income": income,
        "monthly_expenses": expenses,
        "existing_loans": random.randint(0, 5),
        "credit_score": credit_score,
        "loan_amount": loan_amount,
        "loan_purpose": random.choice([
            "Home",
            "Car",
            "Business",
            "Education"
        ]),
        "years_employed": years_employed,
        "repayment_history": repayment_history,
        "dependents": random.randint(0, 5)
    })

df = pd.DataFrame(data)

csv_path = f"{BASE_PATH}/data/applicants.csv"

df.to_csv(csv_path, index=False)

print("Dataset created successfully.")
print(df.head())

Dataset created successfully.
  applicant_id             name  age employment_type  annual_income  \
0     APP-1000  Michael Jimenez   34        Salaried          61007   
1     APP-1001     Cindy Garcia   28   Self-Employed          72267   
2     APP-1002  Vanessa Wiggins   27   Self-Employed         105041   
3     APP-1003      Nathan York   53   Self-Employed          81288   
4     APP-1004        Larry Liu   43        Salaried          98223   

   monthly_expenses  existing_loans  credit_score  loan_amount loan_purpose  \
0              3711               0           594       282085         Home   
1              6545               3           645       229827    Education   
2              4317               2           809       383307    Education   
3              6394               1           664       310085          Car   
4              4278               5           668       346007          Car   

   years_employed repayment_history  dependents  
0               1 

In [ ]:
loan_policy = """
LOAN ELIGIBILITY POLICY

1. Applicants must have a minimum credit score of 650.

2. Applicants with debt-to-income ratio above 45% require manual review.

3. Applicants with annual income below $40,000 are considered high risk.

4. Salaried applicants with stable employment history above 2 years are preferred.

5. Applicants requesting loan amounts greater than five times their annual income may be rejected.

6. Applicants with poor repayment history are considered high risk.

7. Maximum eligible age at loan maturity is 65 years.
"""

with open(f"{BASE_PATH}/data/policies/loan_policy.txt", "w") as f:
    f.write(loan_policy)

print("loan_policy.txt created.")

loan_policy.txt created.


In [ ]:
risk_guidelines = """
RISK ASSESSMENT GUIDELINES

1. Credit scores below 600 indicate high financial risk.

2. Debt-to-income ratio above 55% should lead to rejection.

3. Self-employed applicants with less than 2 years of employment history require manual review.

4. Applicants with multiple existing loans are considered moderate risk.

5. Large loan requests combined with low income increase fraud probability.

6. Inconsistent repayment history increases default likelihood.

7. High monthly expenses relative to income indicate repayment instability.
"""

with open(f"{BASE_PATH}/data/policies/risk_guidelines.txt", "w") as f:
    f.write(risk_guidelines)

print("risk_guidelines.txt created.")

risk_guidelines.txt created.


In [ ]:
manual_review = """
MANUAL REVIEW STANDARD OPERATING PROCEDURE

1. Applications with medium risk scores between 40 and 60 require human review.

2. Missing applicant information should trigger manual verification.

3. Applications flagged for possible fraud must be escalated.

4. Borderline credit scores between 620 and 680 require additional scrutiny.

5. Loan officers may override automated recommendations with justification.

6. Human reviewers must document final approval reasons.
"""

with open(f"{BASE_PATH}/data/policies/manual_review_sop.txt", "w") as f:
    f.write(manual_review)

print("manual_review_sop.txt created.")

manual_review_sop.txt created.


In [ ]:
import os

policy_path = f"{BASE_PATH}/data/policies"

print(os.listdir(policy_path))

['loan_policy.txt', 'risk_guidelines.txt', 'manual_review_sop.txt']


In [ ]:
!pip install -q reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.3 MB/s eta 0:00:00


In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def create_loan_pdf(filename, content):

    path = f"{BASE_PATH}/data/sample_pdfs/{filename}"

    c = canvas.Canvas(path, pagesize=letter)

    text = c.beginText(40, 750)
    text.setFont("Helvetica", 12)

    for line in content.split("\n"):
        text.textLine(line)

    c.drawText(text)
    c.save()

    print(f"{filename} created successfully.")

In [ ]:
approved_case = """
Loan Application Form

Applicant Name: John Doe
Age: 34
Employment Type: Salaried
Annual Income: 95000
Loan Amount Requested: 250000
Credit Score: 780
Existing Loans: 1
Repayment History: Good
Years Employed: 6
"""

create_loan_pdf("approved_case.pdf", approved_case)

approved_case.pdf created successfully.


In [ ]:
rejected_case = """
Loan Application Form

Applicant Name: Mark Smith
Age: 28
Employment Type: Self-Employed
Annual Income: 32000
Loan Amount Requested: 400000
Credit Score: 590
Existing Loans: 4
Repayment History: Poor
Years Employed: 1
"""

create_loan_pdf("rejected_case.pdf", rejected_case)

rejected_case.pdf created successfully.


In [ ]:
review_case = """
Loan Application Form

Applicant Name: Sarah Johnson
Age: 40
Employment Type: Self-Employed
Annual Income: 65000
Loan Amount Requested: 180000
Credit Score: 670
Existing Loans: 2
Repayment History: Average
Years Employed: 2
"""

create_loan_pdf("review_case.pdf", review_case)

review_case.pdf created successfully.


In [ ]:
invalid_case = """
Loan Application Form

Applicant Name: Unknown
Annual Income: Not Provided
Loan Amount Requested: 500000
"""

create_loan_pdf("invalid_case.pdf", invalid_case)

invalid_case.pdf created successfully.


In [ ]:
pdf_path = f"{BASE_PATH}/data/sample_pdfs"

print(os.listdir(pdf_path))

['rejected_case.pdf', 'approved_case.pdf', 'review_case.pdf', 'invalid_case.pdf']


I created synthetic banking policy documents including:


*   Loan approval rules
*   Risk classification guidelines
*   Manual review procedures








These documents simulate real banking compliance policies.